# 01 · 鸢尾花分类：第一个完整的机器学习流程

这是最经典的入门例子。流程只有五步，之后所有练习都是这五步的变体：

1. 拿数据
2. 分成训练集和测试集
3. 选模型、训练
4. 预测
5. 评估

**运行方式**：选中单元格，按 `Shift + Enter`，单元格在原地执行，结果显示在下面。

In [ ]:
import matplotlib
matplotlib.rcParams["font.sans-serif"] = ["Noto Sans CJK SC", "WenQuanYi Zen Hei", "DejaVu Sans"]
matplotlib.rcParams["axes.unicode_minus"] = False
print("中文字体:", matplotlib.rcParams["font.sans-serif"][0])


In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_iris

iris = load_iris()
X, y = iris.data, iris.target

print("特征形状:", X.shape)
print("标签形状:", y.shape)
print("类别:", iris.target_names)
print("特征名:", iris.feature_names)

`X` 是 150 行 × 4 列——150 朵花，每朵量了 4 个尺寸。`y` 是每朵花的品种编号（0/1/2）。

先看一眼数据长什么样。`display()` 比 `print()` 好看，在 Jupyter 里会渲染成表格。

In [ ]:
df = pd.DataFrame(X, columns=iris.feature_names)
df["品种"] = [iris.target_names[i] for i in y]
display(df.head())
display(df["品种"].value_counts())

## 切分数据

`train_test_split` 把数据随机分成两份：训练用的和考试用的。

`random_state=42` 是随机种子，固定它才能每次跑出一样的结果——调试时很重要，否则你永远不知道变好是因为改了代码还是因为运气。

`stratify=y` 保证训练集和测试集里三个品种的比例一致，避免测试集恰好全是同一类。

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"训练集 {X_train.shape[0]} 条，测试集 {X_test.shape[0]} 条")

## 训练模型

先上最朴素的**逻辑回归**打底。名字里有"回归"，实际是分类器。

sklearn 的接口极其统一，记住三个方法就够用一辈子：

- `.fit(X, y)` 训练
- `.predict(X)` 预测
- `.score(X, y)` 算准确率

In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=200)
model.fit(X_train, y_train)

print(f"训练集准确率: {model.score(X_train, y_train):.3f}")
print(f"测试集准确率: {model.score(X_test, y_test):.3f}")

## 评估

准确率只是一个数字。`classification_report` 告诉你**具体错在哪**——哪一类召回低、哪两类被混淆了。

混淆矩阵是矩阵形式：行是真实类别，列是预测类别，对角线是猜对的。

In [ ]:
from sklearn.metrics import classification_report, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred, target_names=iris.target_names))

ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred, display_labels=iris.target_names, cmap="Blues"
)
plt.title("混淆矩阵")
plt.show()

## 换个模型对比

同样的数据、同样的切分，换一个算法看看。**这一步是机器学习里最常做的事**——不是调参调到底，而是先横向比几个模型看谁合适。

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

models = {
    "逻辑回归": LogisticRegression(max_iter=200),
    "KNN (k=5)": KNeighborsClassifier(n_neighbors=5),
    "SVM (RBF)": SVC(),
    "随机森林": RandomForestClassifier(n_estimators=100, random_state=42),
}

for name, m in models.items():
    m.fit(X_train, y_train)
    print(f"{name:<12} 测试集准确率 {m.score(X_test, y_test):.3f}")

## 留给你动手的

改代码跑一下，观察结果变化：

1. 把 `test_size` 改成 `0.5`，准确率怎么变？为什么变小了？
2. 把 `random_state` 换成 `0`，结果还一样吗？差多少？
3. 去掉 `stratify=y` 试试，会不会有一次崩掉？
4. `KNeighborsClassifier` 的 `n_neighbors` 从 1 试到 20，画一条准确率曲线——这就是**超参数搜索**的雏形。

> 提示：第 4 题可以用列表推导式一次算完，不用手改十遍。